# Unified Observability: App, GPU & LLM Monitoring + Tracing

This notebook deploys a complete observability stack that provides:

| Layer | What | Tool |
|-------|------|------|
| **App Metrics** | Request rate, latency, error rate, active orders | Prometheus + Grafana |
| **GPU Metrics** | Utilization, memory, power, temperature | DCGM Exporter + Grafana |
| **LLM Metrics** | Throughput (tok/s), TTFT, ITL, queue depth, cache | vLLM built-in + Grafana |
| **LLM Tracing** | End-to-end request traces through the inference stack | OpenTelemetry + Jaeger |

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      Metric Sources                             │
│  cafe-api /metrics    vLLM /metrics    DCGM Exporter           │
│       │                    │                │                   │
└───────┼────────────────────┼────────────────┼───────────────────┘
        └──────────┬─────────┘────────────────┘
                   ▼
    ┌──────────────────────────────┐
    │  OpenShift User Workload     │
    │  Monitoring (Prometheus)     │
    └──────────────┬───────────────┘
                   ▼
    ┌──────────────────────────────┐      ┌──────────────────┐
    │       Thanos Querier         │      │  OTel Collector   │
    └──────────────┬───────────────┘      └────────┬─────────┘
                   │                               │
                   ▼                               ▼
    ┌──────────────────────────────────────────────────────────┐
    │              Grafana (Dashboards)  +  Jaeger (Traces)    │
    └──────────────────────────────────────────────────────────┘
```

### Prerequisites

- Completed `0_setup/` (cluster, model, cafe-api deployed)
- `oc` CLI logged in with cluster-admin or monitoring permissions
- User Workload Monitoring enabled on the cluster
- NVIDIA GPU Operator installed (for GPU metrics)
- `.env` file configured with `CLUSTER_DOMAIN`, `MODEL_ENDPOINT`, `MODEL_NAME`, `MAAS_API_KEY`

## 1. Prerequisites Check

In [ ]:
import subprocess, json, os, sys
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

if not CLUSTER_DOMAIN:
    result = subprocess.run(
        ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    CLUSTER_DOMAIN = result.stdout.strip()

print(f"Cluster Domain: {CLUSTER_DOMAIN}")
print(f"Model Namespace: {MODEL_NAMESPACE}")
print(f"Model Endpoint: {MODEL_ENDPOINT}")
print(f"Model Name: {MODEL_NAME}")
print(f"MaaS API Key: {'configured' if MAAS_API_KEY else '⚠️  not set'}")

In [ ]:
%%bash
echo "=== Checking prerequisites ==="
echo ""

# 1. oc login
echo -n "[1/5] oc login: "
if oc whoami &>/dev/null; then
    echo "✅ $(oc whoami)"
else
    echo "❌ Not logged in"
    exit 1
fi

# 2. User Workload Monitoring
echo -n "[2/5] User Workload Monitoring: "
UWM_PODS=$(oc get pods -n openshift-user-workload-monitoring --no-headers 2>/dev/null | wc -l)
if [ "$UWM_PODS" -gt 0 ]; then
    echo "✅ ($UWM_PODS pods running)"
else
    echo "❌ Not enabled — enable via cluster-monitoring-config ConfigMap"
fi

# 3. GPU Operator / DCGM
echo -n "[3/5] NVIDIA GPU Operator: "
GPU_NS=$(oc get ns nvidia-gpu-operator --no-headers 2>/dev/null | awk '{print $1}')
if [ -n "$GPU_NS" ]; then
    DCGM_PODS=$(oc get pods -n nvidia-gpu-operator -l app=nvidia-dcgm-exporter --no-headers 2>/dev/null | wc -l)
    echo "✅ (DCGM exporter: ${DCGM_PODS} pods)"
else
    echo "⚠️  Namespace not found — GPU metrics will be unavailable"
fi

# 4. vLLM model serving (llm-d uses app.kubernetes.io/part-of label)
echo -n "[4/5] vLLM Model Serving: "
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL_PODS=$(oc get pods -n $MODEL_NS -l app.kubernetes.io/part-of=llminferenceservice --no-headers 2>/dev/null | grep Running | wc -l)
if [ "$MODEL_PODS" -gt 0 ]; then
    echo "✅ ($MODEL_PODS pods running in $MODEL_NS)"
else
    echo "⚠️  No running inference pods in $MODEL_NS namespace"
fi

# 5. Cafe API
echo -n "[5/5] Cafe API: "
CAFE_PODS=$(oc get pods -n cafe-system -l app=cafe-api --no-headers 2>/dev/null | grep Running | wc -l)
if [ "$CAFE_PODS" -gt 0 ]; then
    echo "✅ ($CAFE_PODS pods running)"
else
    echo "⚠️  cafe-api not running — run 0_setup/2_app_setup.ipynb first"
fi

echo ""
echo "=== Prerequisites check complete ==="

## 2. Instrument Cafe App (Application Metrics + Tracing)

We add two capabilities to the cafe-order-system:
1. **Prometheus `/metrics` endpoint** — request count, latency histogram, active orders gauge
2. **OpenTelemetry tracing** — distributed traces exported to the OTel Collector

The instrumented source code is already prepared in `../0_setup/apps/cafe-order-system/`. We rebuild the image and redeploy.

In [ ]:
%%bash
echo "=== Rebuilding cafe-api with metrics + tracing instrumentation ==="
echo ""

cd ../0_setup

# Rebuild image
echo "Starting build..."
oc start-build cafe-api \
    --from-dir=apps/cafe-order-system \
    -n cafe-system \
    --follow --wait

echo ""
echo "Restarting deployment..."
oc rollout restart deploy/cafe-api -n cafe-system
oc wait --for=condition=available deployment/cafe-api -n cafe-system --timeout=120s

echo ""
echo "✅ cafe-api rebuilt with instrumentation"

In [ ]:
%%bash
echo "=== Verifying /metrics endpoint ==="
ROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath='{.spec.host}')
echo "Route: http://${ROUTE}"
echo ""

# Send a few requests to generate metrics
curl -s "http://${ROUTE}/health" > /dev/null
curl -s "http://${ROUTE}/api/menu/" > /dev/null
curl -s "http://${ROUTE}/api/customers/" > /dev/null

echo "Prometheus metrics (sample):"
curl -s "http://${ROUTE}/metrics" | grep -E '^cafe_' | head -20

In [ ]:
%%bash
echo "=== Deploying ServiceMonitor for cafe-api ==="

# Ensure the service has the correct label and port name
oc label svc/cafe-api -n cafe-system app=cafe-api --overwrite
oc patch svc cafe-api -n cafe-system --type='json' \
    -p='[{"op": "replace", "path": "/spec/ports/0/name", "value": "http"}]' 2>/dev/null || true

oc apply -f manifests/01-servicemonitor-cafe.yaml
echo ""
echo "✅ ServiceMonitor created — Prometheus will scrape cafe-api /metrics"

## 3. LLM Metrics (vLLM)

vLLM exposes Prometheus metrics **by default** on the same port as the inference API (HTTPS on port 8000). Key metrics include:

| Metric | Description |
|--------|-------------|
| `vllm:num_requests_running` | Currently processing requests |
| `vllm:num_requests_waiting` | Queued requests |
| `vllm:avg_generation_throughput_toks_per_s` | Token generation throughput |
| `vllm:time_to_first_token_seconds` | Time to first token (TTFT) |
| `vllm:time_per_output_token_seconds` | Inter-token latency (ITL) |
| `vllm:kv_cache_usage_perc` | KV cache utilization |
| `vllm:prompt_tokens_total` | Total prompt tokens processed |
| `vllm:generation_tokens_total` | Total tokens generated |

Note: llm-d (LLMInferenceService) pods use label `app.kubernetes.io/part-of=llminferenceservice` and serve metrics over **HTTPS** with self-signed TLS.

In [ ]:
%%bash
echo "=== Checking vLLM metrics availability ==="
echo ""

MODEL_NS=${MODEL_NAMESPACE:-demo}

# Find vLLM pod (llm-d label)
VLLM_POD=$(oc get pods -n $MODEL_NS -l app.kubernetes.io/part-of=llminferenceservice \
    --field-selector=status.phase=Running -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$VLLM_POD" ]; then
    echo "⚠️  No vLLM pods found in namespace $MODEL_NS"
    echo "   Deploy a model first via 0_setup/1_environment_setup.ipynb"
    exit 0
fi

echo "vLLM Pod: $VLLM_POD"
echo ""

# Check metrics via service (HTTPS)
SVC_NAME=$(oc get svc -n $MODEL_NS -l app.kubernetes.io/part-of=llminferenceservice \
    -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
echo "Service: $SVC_NAME"
echo ""
echo "Sample vLLM metrics (via temporary pod):"
oc run vllm-metrics-check --rm -i --restart=Never --image=curlimages/curl:latest -n $MODEL_NS \
    -- curl -sk https://${SVC_NAME}.${MODEL_NS}.svc.cluster.local:8000/metrics 2>/dev/null \
    | grep '^vllm:' | head -10 || echo "  (will be accessible after PodMonitor is deployed)"

In [ ]:
%%bash
echo "=== Deploying PodMonitor for vLLM ==="

oc apply -f manifests/02-podmonitor-vllm.yaml
echo ""
echo "✅ PodMonitor created — Prometheus will scrape vLLM inference pods (HTTPS)"

## 4. GPU Metrics (NVIDIA DCGM)

The NVIDIA GPU Operator deploys the **DCGM Exporter** as a DaemonSet on GPU nodes. It exposes metrics such as:

| Metric | Description |
|--------|-------------|
| `DCGM_FI_DEV_GPU_UTIL` | GPU utilization (%) |
| `DCGM_FI_DEV_FB_USED` | Frame buffer memory used (MiB) |
| `DCGM_FI_DEV_FB_FREE` | Frame buffer memory free (MiB) |
| `DCGM_FI_DEV_POWER_USAGE` | Power draw (W) |
| `DCGM_FI_DEV_GPU_TEMP` | Temperature (C) |
| `DCGM_FI_DEV_SM_CLOCK` | Streaming Multiprocessor clock (MHz) |

These are typically already scraped by the platform Prometheus via the GPU Operator's ServiceMonitor.

In [ ]:
%%bash
echo "=== Verifying DCGM Exporter ==="
echo ""

GPU_NS="nvidia-gpu-operator"
if ! oc get ns $GPU_NS &>/dev/null; then
    echo "⚠️  Namespace $GPU_NS not found"
    GPU_NS=$(oc get pods --all-namespaces -l app=nvidia-dcgm-exporter -o jsonpath='{.items[0].metadata.namespace}' 2>/dev/null)
    if [ -z "$GPU_NS" ]; then
        echo "❌ DCGM Exporter not found on this cluster"
        echo "   GPU metrics will not be available in Grafana"
        exit 0
    fi
fi

echo "GPU Operator namespace: $GPU_NS"
echo ""
echo "DCGM Exporter pods:"
oc get pods -n $GPU_NS -l app=nvidia-dcgm-exporter --no-headers 2>/dev/null || \
    oc get pods -n $GPU_NS -l app.kubernetes.io/name=dcgm-exporter --no-headers 2>/dev/null || \
    echo "  (pods not found with expected labels)"

echo ""
echo "Checking if DCGM metrics are already scraped..."
SM=$(oc get servicemonitor -n $GPU_NS -o name 2>/dev/null | grep -i dcgm | head -1)
if [ -n "$SM" ]; then
    echo "✅ ServiceMonitor already exists: $SM"
    echo "   GPU metrics are being collected by Prometheus"
else
    echo "⚠️  No DCGM ServiceMonitor found — GPU Operator usually configures this automatically"
fi

## 5. LLM Call Tracing (OpenTelemetry + Jaeger)

Deploy the distributed tracing stack:
1. **OpenTelemetry Collector** — receives OTLP traces from instrumented apps
2. **Jaeger** — stores and visualizes traces

The cafe-api app is already instrumented with OpenTelemetry (via the rebuild in Section 2). We just need to set the `OTEL_EXPORTER_OTLP_ENDPOINT` environment variable on the deployment.

In [ ]:
%%bash
echo "=== Creating monitoring namespace ==="
oc apply -f manifests/00-namespace.yaml
echo ""

echo "=== Deploying Jaeger (all-in-one) ==="
oc apply -f manifests/04-jaeger.yaml
echo ""

echo "=== Deploying OpenTelemetry Collector ==="
oc apply -f manifests/03-otel-collector.yaml
echo ""

echo "Waiting for deployments..."
oc wait --for=condition=available deployment/jaeger -n monitoring --timeout=120s
oc wait --for=condition=available deployment/otel-collector -n monitoring --timeout=120s

echo ""
echo "✅ Tracing stack deployed"

In [ ]:
%%bash
echo "=== Configuring cafe-api to send traces ==="

# Set OTEL endpoint on cafe-api deployment
oc set env deploy/cafe-api -n cafe-system \
    OTEL_EXPORTER_OTLP_ENDPOINT=otel-collector.monitoring.svc.cluster.local:4317

# Wait for rollout
oc rollout status deploy/cafe-api -n cafe-system --timeout=60s

echo ""
echo "✅ cafe-api configured to export traces to OTel Collector"

## 6. Grafana Dashboards (Unified View)

Deploy Grafana with pre-configured datasources and dashboards:
- **Prometheus** datasource → queries Thanos Querier for app/GPU/LLM metrics
- **Jaeger** datasource → correlates traces with metrics
- Pre-built dashboards: App, GPU, LLM

In [ ]:
%%bash
echo "=== Creating Grafana dashboard ConfigMaps ==="

oc create configmap grafana-dashboard-app \
    --from-file=app-dashboard.json=manifests/07-dashboard-app.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

oc create configmap grafana-dashboard-gpu \
    --from-file=gpu-dashboard.json=manifests/08-dashboard-gpu.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

oc create configmap grafana-dashboard-llm \
    --from-file=llm-dashboard.json=manifests/09-dashboard-llm.json \
    -n monitoring --dry-run=client -o yaml | oc apply -f -

echo "✅ Dashboard ConfigMaps created"

In [ ]:
%%bash
echo "=== Deploying Grafana ==="

# Deploy Grafana (includes ServiceAccount, Deployment, Service, Route)
oc apply -f manifests/05-grafana.yaml

# Grant monitoring view permission to grafana SA
oc adm policy add-cluster-role-to-user cluster-monitoring-view \
    -z grafana -n monitoring 2>/dev/null || true

# Create SA token for Prometheus auth
GRAFANA_TOKEN=$(oc create token grafana -n monitoring --duration=8760h 2>/dev/null || echo "")

# Apply datasources config
oc apply -f manifests/06-grafana-datasources.yaml

if [ -n "$GRAFANA_TOKEN" ]; then
    # Patch datasource configmap with real token
    oc get cm grafana-datasources -n monitoring -o yaml | \
        sed "s|\${GRAFANA_SA_TOKEN}|$GRAFANA_TOKEN|" | \
        oc apply -f -
    echo "✅ Grafana SA token configured for Prometheus access"
else
    echo "⚠️  Could not create token — Grafana may need manual Prometheus auth"
fi

# Wait and restart to pick up all configmaps
oc wait --for=condition=available deployment/grafana -n monitoring --timeout=120s
oc rollout restart deploy/grafana -n monitoring
oc wait --for=condition=available deployment/grafana -n monitoring --timeout=120s

echo ""
echo "✅ Grafana deployed and configured"

## 7. Verification & Demo

Generate traffic (both app and LLM requests) and verify all metrics/traces are flowing.

In [ ]:
%%bash
echo "=== Generating test traffic (Cafe API) ==="
ROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath='{.spec.host}')

for i in $(seq 1 10); do
    curl -s "http://${ROUTE}/api/menu/" > /dev/null
    curl -s "http://${ROUTE}/api/customers/" > /dev/null
    curl -s -X POST "http://${ROUTE}/api/orders/" \
        -H "Content-Type: application/json" \
        -d '{"customer_id": 1, "items": [{"menu_item_id": 1, "quantity": 1}]}' > /dev/null
done

echo "✅ Sent 10 rounds of requests to cafe-api"

In [ ]:
import os, json, subprocess
from dotenv import load_dotenv

load_dotenv("../.env")

MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_NAME = os.getenv("MODEL_NAME", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

print("=== Generating LLM traffic (for vLLM metrics) ===")
print()

success = False

# Method 1: Try MaaS gateway endpoint
if MODEL_ENDPOINT and MAAS_API_KEY:
    import httpx
    print(f"[Method 1] MaaS Gateway: {MODEL_ENDPOINT}")
    try:
        resp = httpx.post(
            f"{MODEL_ENDPOINT}/v1/chat/completions",
            headers={"Authorization": f"Bearer {MAAS_API_KEY}"},
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": "Say hello in Korean."}],
                "max_tokens": 50
            },
            timeout=30,
            verify=False
        )
        if resp.status_code == 200:
            content = resp.json()["choices"][0]["message"]["content"][:60]
            print(f"  ✅ Response: {content}...")
            success = True
        else:
            print(f"  ⚠️  HTTP {resp.status_code} — trying in-cluster fallback")
    except Exception as e:
        print(f"  ⚠️  {e} — trying in-cluster fallback")
    print()

# Method 2: In-cluster direct call via oc exec
if not success:
    print(f"[Method 2] In-cluster direct call to vLLM workload service")
    svc_result = subprocess.run(
        ["oc", "get", "svc", "-n", MODEL_NAMESPACE,
         "-l", "app.kubernetes.io/component=llminferenceservice-workload",
         "-o", "jsonpath={.items[0].metadata.name}"],
        capture_output=True, text=True
    )
    svc_name = svc_result.stdout.strip()
    if svc_name:
        cmd = (
            f'curl -sk -X POST https://{svc_name}.{MODEL_NAMESPACE}.svc.cluster.local:8000/v1/chat/completions '
            f'-H "Content-Type: application/json" '
            f'-d \'{{"model":"{MODEL_NAME}","messages":[{{"role":"user","content":"Hi"}}],"max_tokens":20}}\''
        )
        for i in range(3):
            result = subprocess.run(
                ["oc", "run", f"llm-test-{i}", "--rm", "-i", "--restart=Never",
                 "--image=curlimages/curl:latest", "-n", MODEL_NAMESPACE,
                 "--", "sh", "-c", cmd],
                capture_output=True, text=True, timeout=30
            )
            if "choices" in result.stdout:
                try:
                    data = json.loads(result.stdout)
                    content = data["choices"][0]["message"]["content"][:60]
                    print(f"  [{i+1}/3] ✅ {content}...")
                    success = True
                except:
                    print(f"  [{i+1}/3] ✅ Got response")
                    success = True
            else:
                print(f"  [{i+1}/3] ⚠️  {result.stdout[:80] or result.stderr[:80]}")
    else:
        print("  ⚠️  No workload service found")

print()
if success:
    print("✅ LLM traffic generated — vLLM metrics (TTFT, throughput, etc.) will update")
else:
    print("⚠️  Could not generate LLM traffic via API")
    print("   vLLM metrics are still collected — use IDE/other clients to trigger inference")
    print("   Gauge metrics (cache usage, running requests) are available regardless")

In [ ]:
import subprocess, json, os, urllib.request, urllib.parse, ssl
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
                      capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

THANOS_URL = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"
TOKEN = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def query_prometheus(promql: str) -> dict:
    url = f"{THANOS_URL}/api/v1/query?query={urllib.parse.quote(promql)}"
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())

print("=== Prometheus Metric Verification ===")
print()

ns = MODEL_NAMESPACE
checks = [
    ("Cafe App — request count", f'sum(cafe_http_requests_total{{namespace="cafe-system"}})'),
    ("Cafe App — avg latency (ms)", f'avg(rate(cafe_http_request_duration_seconds_sum{{namespace="cafe-system"}}[5m]) / rate(cafe_http_request_duration_seconds_count{{namespace="cafe-system"}}[5m])) * 1000'),
    ("vLLM — running requests", f'vllm:num_requests_running{{namespace="{ns}"}}'),
    ("vLLM — KV cache usage (%)", f'vllm:kv_cache_usage_perc{{namespace="{ns}"}} * 100'),
    ("GPU — utilization (%)", 'DCGM_FI_DEV_GPU_UTIL'),
    ("GPU — memory used (MiB)", 'DCGM_FI_DEV_FB_USED'),
]

for label, promql in checks:
    try:
        result = query_prometheus(promql)
        data = result.get("data", {}).get("result", [])
        if data:
            value = data[0].get("value", ["", "N/A"])[1]
            try:
                value = f"{float(value):.2f}"
            except (ValueError, TypeError):
                pass
            print(f"  ✅ {label}: {value}")
        else:
            print(f"  ⚠️  {label}: no data yet (may take 1-2 min for first scrape)")
    except Exception as e:
        print(f"  ❌ {label}: {e}")

In [ ]:
import subprocess, os
from dotenv import load_dotenv

load_dotenv("../.env")
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
                      capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

grafana_host = subprocess.run(
    ["oc", "get", "route", "grafana", "-n", "monitoring", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

jaeger_host = subprocess.run(
    ["oc", "get", "route", "jaeger-ui", "-n", "monitoring", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

cafe_host = subprocess.run(
    ["oc", "get", "route", "cafe-api", "-n", "cafe-system", "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True
).stdout.strip()

print("="*70)
print("  MONITORING STACK — ACCESS INFORMATION")
print("="*70)
print()
print(f"  Grafana:       https://{grafana_host}")
print(f"  Jaeger UI:     https://{jaeger_host}")
print(f"  Cafe API:      http://{cafe_host}")
print(f"  Cafe Metrics:  http://{cafe_host}/metrics")
print()
print("-"*70)
print("  Grafana Credentials:  admin / admin")
print("-"*70)
print()
print("  Grafana Dashboards:")
print(f"    • App:  https://{grafana_host}/d/cafe-app-metrics")
print(f"    • GPU:  https://{grafana_host}/d/gpu-dcgm-metrics")
print(f"    • LLM:  https://{grafana_host}/d/llm-vllm-metrics")
print()
print("  Jaeger Tracing:")
print(f"    • Search service: cafe-order-system")
print(f"    • URL: https://{jaeger_host}/search?service=cafe-order-system")
print()
print("="*70)

## 8. Summary

The complete observability stack is now deployed:

| Component | Namespace | Purpose |
|-----------|-----------|----------|
| ServiceMonitor (cafe-api) | `cafe-system` | Scrape app metrics |
| PodMonitor (vLLM) | `demo` | Scrape LLM inference metrics (HTTPS) |
| DCGM Exporter | `nvidia-gpu-operator` | GPU metrics (pre-existing) |
| OpenTelemetry Collector | `monitoring` | Receive & forward traces |
| Jaeger | `monitoring` | Trace storage & UI |
| Grafana | `monitoring` | Unified dashboards |

### What You Can See

| Dashboard | Key Panels |
|-----------|------------|
| **Cafe App** | Request rate, error rate, latency percentiles, active orders |
| **GPU** | Utilization %, memory used/free, power, temperature, clocks |
| **LLM** | Running/waiting requests, tok/s throughput, TTFT, ITL, KV cache usage |
| **Jaeger** | Request traces, span breakdowns, latency distribution |

### Cleanup

```bash
# Remove monitoring stack
oc delete namespace monitoring
oc delete servicemonitor cafe-api -n cafe-system
oc delete podmonitor vllm-inference -n demo

# Remove OTel env from cafe-api
oc set env deploy/cafe-api -n cafe-system OTEL_EXPORTER_OTLP_ENDPOINT-
```

### Next Steps

- Add alerting rules (PrometheusRule CRs) for SLO violations
- Integrate with MaaS gateway metrics (Limitador/Authorino) from `4_control/`
- Enable log correlation with Loki + Grafana